# CS 590 Final Project: Stack Overflow Neo4j Q&A Graph

This notebook loads the Stack Overflow Neo4j-tagged dataset (Users, Tags, Questions, Answers, Comments) into the **stackoverflow** Neo4j database, builds the two relationships recoverable from the source data, validates the load with test queries, and presents an interactive Dash dashboard.

**Data limitation:** the source CSVs provide a meaningless internal row counter for each table (`question_id`, `answer_id`, `comment_id`, `user_id`, `tag_id`). The real Stack Overflow post ID is the `uuid` column for Questions and Answers, and is recoverable from the `link` URL for Comments. Users and Tags have no column or link pattern connecting them to any other table in this export, so they are loaded as unconnected nodes; this is documented as a scoped limitation in the technical report rather than fabricated around.

**Setup:** `database.conf` must be in the same folder as this notebook with a `[neo4j]` section pointing at a database named `stackoverflow`. All five `stackoverflow_nodes_*.csv` files must also be in the same folder.

**Required packages:**
```
pip install neo4j pandas dash plotly
```

## Step 0: Connect to the stackoverflow database

In [1]:
#
# cs590-final-project-etl.ipynb - Loads the Stack Overflow Neo4j-tagged
# dataset (Users, Tags, Questions, Answers, Comments) into the
# stackoverflow Neo4j database, and builds the two relationships that
# are actually recoverable from the source data.
#
# Data limitation documented in the technical report: the CSV exports
# provide a meaningless internal row counter (question_id, answer_id,
# comment_id, user_id, tag_id) alongside a uuid column. For Question
# and Answer, uuid is the real Stack Overflow post ID and matches
# Question.accepted_answer_id directly. For Comment, uuid is the
# comment's own ID, so the comment's parent question must instead be
# recovered by parsing the real question ID out of the comment's link
# URL. User and Tag have no column, link pattern, or uuid that
# connects them to any other table in this export, so they are loaded
# as nodes with no relationships, and that gap is treated as a
# documented limitation rather than something to fabricate around.
#
#------------------------------------------------------------------
# Change History
#------------------------------------------------------------------
# Version   |   Description
#------------------------------------------------------------------
#    1          Initial Development - D. Olaosebikan
#------------------------------------------------------------------

import csv
import html as html_entities
import re
from neo4j import GraphDatabase
from configparser import ConfigParser

db = 'stackoverflow'

def config(filename, section):
    cp = ConfigParser()
    cp.read(filename)
    block = {}
    if cp.has_section(section):
        for item in cp.items(section):
            block[item[0]] = item[1]
    else:
        raise Exception(f"Section {section} was not found in the {filename} file!")
    return block

def dbConnect(filename, section):
    confData = config(filename, section)
    try:
        URI = "neo4j://" + confData["host"] + "/" + confData["database"]
        AUTH = (confData["user"], confData["password"])
        conn = GraphDatabase.driver(URI, auth=AUTH)
        conn.verify_connectivity()
        print("Connection Verified!")
        return conn
    except Exception as error:
        print(error)
        return None

conn = dbConnect('database.conf', 'neo4j')


Connection Verified!


## Step 1: Constraints and indexes

In [2]:
# ------------------------------------------------------------------
# Step 1: Constraints and indexes
#
# A uniqueness constraint on each node label's id property both
# enforces a data integrity rule (no duplicate posts) and creates a
# backing index automatically, which is what makes the MERGE-based
# load below, and later lookups by id, efficient rather than
# requiring a full label scan for every row.
# ------------------------------------------------------------------

constraints = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (u:User) REQUIRE u.userId IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Tag) REQUIRE t.tagId IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (q:Question) REQUIRE q.postId IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (a:Answer) REQUIRE a.postId IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Comment) REQUIRE c.commentId IS UNIQUE",
]
for stmt in constraints:
    conn.execute_query(stmt, database_=db)
print(f'{len(constraints)} constraints/indexes created or already present.')


5 constraints/indexes created or already present.


## Step 2: Load nodes from CSV

In [4]:
# ------------------------------------------------------------------
# Step 2: Load nodes
#
# Each loader reads its CSV and MERGEs one node per row, keyed on the
# real, stable identifier discussed above. MERGE makes every loader
# safe to re-run without creating duplicate nodes.
# ------------------------------------------------------------------

def loadUsers(path, conn):
    """Loads each User row, keyed on the real Stack Overflow user id
    in the uuid column. Two non-numeric sentinel values appear in
    this export: 'deleted', used for accounts Stack Overflow has
    removed, and '-1', the platform's own placeholder for the
    'Community' pseudo-user that owns community-wiki content. '-1'
    parses fine as an integer and is loaded normally; 'deleted' does
    not, so those rows are skipped and counted rather than allowed to
    raise a null-property error on the userId uniqueness constraint."""
    cypher = """
    MERGE (u:User {userId: toInteger($uuid)})
    SET u.displayName = $displayName
    """
    count = 0
    skipped = 0
    with open(path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            uuid_val = row['uuid'].strip()
            if not uuid_val.lstrip('-').isdigit():
                skipped += 1
                continue
            conn.execute_query(cypher, uuid=uuid_val,
                                displayName=row['display_name'], database_=db)
            count += 1
    return count, skipped

def loadTags(path, conn):
    cypher = """
    MERGE (t:Tag {tagId: toInteger($tagId)})
    SET t.name = $name, t.link = $link
    """
    count = 0
    with open(path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            conn.execute_query(cypher, tagId=row['tag_id'], name=row['name'],
                                link=row['link'], database_=db)
            count += 1
    return count

def loadQuestions(path, conn):
    """Loads each Question row. Stack Overflow's export encodes
    special characters (apostrophes, quotation marks) as HTML
    entities, e.g. doesn&#39;t instead of doesn't, so titles are
    decoded with html_entities.unescape() before being stored,
    otherwise they display literally as escaped markup in any later
    view. The standard library html module is imported here as
    html_entities rather than html, because the dashboard cell later
    in this notebook imports Dash's own html module (used for
    html.Div, html.H2, etc.) under the plain name html; since all
    cells in a notebook share one namespace, importing the standard
    library module under its own name would let the later Dash
    import silently shadow it, which is what originally caused an
    AttributeError when this loader ran after the dashboard cell."""
    cypher = """
    MERGE (q:Question {postId: toInteger($postId)})
    SET q.title = $title, q.link = $link,
        q.creationDate = datetime({epochSeconds: toInteger($creationDate)}),
        q.viewCount = toInteger($viewCount),
        q.answerCount = toInteger($answerCount),
        q.acceptedAnswerId = CASE WHEN $acceptedAnswerId = '' THEN null
                                   ELSE toInteger($acceptedAnswerId) END
    """
    count = 0
    with open(path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            conn.execute_query(
                cypher, postId=row['uuid'], title=html_entities.unescape(row['title']), link=row['link'],
                creationDate=row['creation_date'], viewCount=row['view_count'],
                answerCount=row['answer_count'],
                acceptedAnswerId=row['accepted_answer_id'], database_=db
            )
            count += 1
    return count

def loadAnswers(path, conn):
    """Loads each Answer row, applying the same html_entities.unescape()
    fix to the title field as loadQuestions(), since Answer titles
    come from the same Stack Overflow export and carry the same
    encoded entities."""
    cypher = """
    MERGE (a:Answer {postId: toInteger($postId)})
    SET a.title = $title, a.link = $link,
        a.isAccepted = ($isAccepted = 'true'),
        a.score = toInteger($score)
    """
    count = 0
    with open(path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            conn.execute_query(
                cypher, postId=row['uuid'], title=html_entities.unescape(row['title']), link=row['link'],
                isAccepted=row['is_accepted'], score=row['score'], database_=db
            )
            count += 1
    return count

# Matches the real question id embedded in a comment's permalink, e.g.
# https://stackoverflow.com/questions/69208813/...#comment122336972_69208813
COMMENT_QUESTION_ID = re.compile(r'stackoverflow\.com/questions/(\d+)/')

def loadComments(path, conn):
    cypher = """
    MERGE (c:Comment {commentId: toInteger($commentId)})
    SET c.score = toInteger($score), c.link = $link,
        c.parentQuestionId = $parentQuestionId
    """
    count = 0
    unmatched = 0
    with open(path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            m = COMMENT_QUESTION_ID.search(row['link'])
            parent_id = m.group(1) if m else None
            if parent_id is None:
                unmatched += 1
            conn.execute_query(
                cypher, commentId=row['uuid'], score=row['score'], link=row['link'],
                parentQuestionId=parent_id, database_=db
            )
            count += 1
    return count, unmatched

users_loaded, users_skipped = loadUsers('stackoverflow.nodes.User.csv', conn)
tags_loaded = loadTags('stackoverflow.nodes.Tag.csv', conn)
questions_loaded = loadQuestions('stackoverflow.nodes.Question.csv', conn)
answers_loaded = loadAnswers('stackoverflow.nodes.Answer.csv', conn)
comments_loaded, comments_unmatched = loadComments('stackoverflow.nodes.Comment.csv', conn)

print(f'Users loaded: {users_loaded} ({users_skipped} skipped: non-numeric id, e.g. "deleted")')
print(f'Tags loaded: {tags_loaded}')
print(f'Questions loaded: {questions_loaded}')
print(f'Answers loaded: {answers_loaded}')
print(f'Comments loaded: {comments_loaded} ({comments_unmatched} could not be parsed)')


Users loaded: 1364 (1 skipped: non-numeric id, e.g. "deleted")
Tags loaded: 476
Questions loaded: 1589
Answers loaded: 1367
Comments loaded: 1396 (0 could not be parsed)


## Step 3: Build relationships

In [5]:
# ------------------------------------------------------------------
# Step 3: Build relationships
#
# Only two relationship types are recoverable from this export:
#   Question -[:HAS_ACCEPTED_ANSWER]-> Answer
#   Comment  -[:COMMENTS_ON]-> Question
# Both are built from a MATCH...MATCH...MERGE pattern rather than
# during node loading, since the related node may not have been
# loaded yet at the point a given row is processed.
# ------------------------------------------------------------------

def linkAcceptedAnswers(conn):
    cypher = """
    MATCH (q:Question), (a:Answer)
    WHERE q.acceptedAnswerId = a.postId
    MERGE (q)-[:HAS_ACCEPTED_ANSWER]->(a)
    RETURN count(*) AS linked
    """
    records, summary, keys = conn.execute_query(cypher, database_=db)
    return records[0]['linked']

def linkCommentsToQuestions(conn):
    cypher = """
    MATCH (c:Comment), (q:Question)
    WHERE c.parentQuestionId = toString(q.postId)
    MERGE (c)-[:COMMENTS_ON]->(q)
    RETURN count(*) AS linked
    """
    records, summary, keys = conn.execute_query(cypher, database_=db)
    return records[0]['linked']

accepted_links = linkAcceptedAnswers(conn)
comment_links = linkCommentsToQuestions(conn)

print(f'\nQuestion-[:HAS_ACCEPTED_ANSWER]->Answer relationships created: {accepted_links}')
print(f'Comment-[:COMMENTS_ON]->Question relationships created: {comment_links}')



Question-[:HAS_ACCEPTED_ANSWER]->Answer relationships created: 619
Comment-[:COMMENTS_ON]->Question relationships created: 1396


## Step 4: Test queries (validate the load)

In [6]:
#
# cs590-final-project-queries-dashboard.ipynb (continued) - Test
# queries that validate the data load, and an interactive Dash
# dashboard built on top of the stackoverflow Neo4j database.
#

import pandas as pd

# ------------------------------------------------------------------
# Step 4: Test queries
#
# Each query validates one part of the load: that node counts match
# the source CSVs, that the two relationship types were built
# correctly, and that the data needed for the dashboard is queryable.
# ------------------------------------------------------------------

def testNodeCounts(conn):
    cypher = """
    MATCH (n)
    RETURN labels(n)[0] AS label, count(n) AS total
    ORDER BY label
    """
    records, summary, keys = conn.execute_query(cypher, database_=db)
    return pd.DataFrame([dict(r) for r in records])

def testAcceptedAnswerSample(conn):
    cypher = """
    MATCH (q:Question)-[:HAS_ACCEPTED_ANSWER]->(a:Answer)
    RETURN q.postId AS questionId, q.title AS questionTitle,
           a.postId AS answerId, a.score AS answerScore
    ORDER BY answerScore DESC
    LIMIT 5
    """
    records, summary, keys = conn.execute_query(cypher, database_=db)
    return pd.DataFrame([dict(r) for r in records])

def testCommentsPerQuestion(conn):
    cypher = """
    MATCH (c:Comment)-[:COMMENTS_ON]->(q:Question)
    RETURN q.postId AS questionId, q.title AS questionTitle, count(c) AS commentCount
    ORDER BY commentCount DESC
    LIMIT 5
    """
    records, summary, keys = conn.execute_query(cypher, database_=db)
    return pd.DataFrame([dict(r) for r in records])

def testOrphanRate(conn):
    """Reports what fraction of Questions and Answers have no
    relationship at all, quantifying the dataset limitation discussed
    in the technical report rather than leaving it as an assertion."""
    cypher = """
    MATCH (q:Question)
    WITH count(q) AS totalQuestions
    MATCH (q2:Question)-[:HAS_ACCEPTED_ANSWER]->()
    WITH totalQuestions, count(q2) AS questionsWithAcceptedAnswer
    MATCH (a:Answer)
    WITH totalQuestions, questionsWithAcceptedAnswer, count(a) AS totalAnswers
    MATCH (a2:Answer)<-[:HAS_ACCEPTED_ANSWER]-()
    RETURN totalQuestions, questionsWithAcceptedAnswer,
           totalAnswers, count(a2) AS answersLinkedToQuestion
    """
    records, summary, keys = conn.execute_query(cypher, database_=db)
    return dict(records[0])

print('Node counts by label:')
print(testNodeCounts(conn))

print('\nTop 5 accepted answers by score:')
print(testAcceptedAnswerSample(conn))

print('\nTop 5 most-commented questions:')
print(testCommentsPerQuestion(conn))

print('\nRelationship coverage (documents the dataset limitation):')
print(testOrphanRate(conn))


Node counts by label:
      label  total
0    Answer   1367
1   Comment   1396
2  Question   1589
3       Tag    476
4      User   1364

Top 5 accepted answers by score:
   questionId                                      questionTitle  answerId  \
0    61446845       Neo4j Filter function is no longer supported  61447621   
1    68655431  Neo4J Cypher: Set property only if it is null ...  68655822   
2    67701558  Creating node and set ID for it without duplic...  67702695   
3    67459833              Neo4j poor order by query performance  67470908   
4    65669613  How do I return relationship properties (Neo4j...  65673386   

   answerScore  
0            6  
1            4  
2            4  
3            4  
4            4  

Top 5 most-commented questions:
   questionId                                      questionTitle  commentCount
0    65873569  neo4j - pagination return same results even wh...            13
1    66395144                     Neo4j WHERE causes duplicates?    

## Step 5: Dashboard queries

In [7]:
# ------------------------------------------------------------------
# Step 5: Dashboard queries
# ------------------------------------------------------------------

def queryQuestionsOverTime(conn):
    """Question volume by creation year, for a time-based view."""
    cypher = """
    MATCH (q:Question)
    RETURN q.creationDate.year AS year, count(q) AS total
    ORDER BY year
    """
    records, summary, keys = conn.execute_query(cypher, database_=db)
    return pd.DataFrame([dict(r) for r in records])

def queryAnswerScoreDistribution(conn):
    """Answer score values, for a distribution view, split by
    whether the answer was the accepted one."""
    cypher = """
    MATCH (a:Answer)
    RETURN a.score AS score, a.isAccepted AS isAccepted
    """
    records, summary, keys = conn.execute_query(cypher, database_=db)
    return pd.DataFrame([dict(r) for r in records])

def queryTopQuestionsByViews(conn):
    """The most-viewed questions, for a ranked table view."""
    cypher = """
    MATCH (q:Question)
    RETURN q.postId AS postId, q.title AS title, q.viewCount AS views,
           q.answerCount AS answers
    ORDER BY views DESC
    LIMIT 100
    """
    records, summary, keys = conn.execute_query(cypher, database_=db)
    return pd.DataFrame([dict(r) for r in records])

df_time = queryQuestionsOverTime(conn)
df_scores = queryAnswerScoreDistribution(conn)
df_top_questions = queryTopQuestionsByViews(conn)

print('\nQuestions by year:')
print(df_time)
print('\nTop questions by view count (first 5 of 100):')
print(df_top_questions.head())



Questions by year:
   year  total
0  2020    534
1  2021   1055

Top questions by view count (first 5 of 100):
     postId                                              title  views  answers
0  66138655  Java Input byte array has wrong 4-byte ending ...   1851        0
1  67030828  Given a plain text and a possible cipher text,...   1820        3
2  63477398  Database 'neo4j' is unavailable. Cannot reset ...   1291        2
3  62365265  Neo.ClientError.Statement.SyntaxError: Invalid...   1201        1
4  61446845       Neo4j Filter function is no longer supported   1072        1


## Step 6: Build and run the interactive dashboard

In [8]:
# ------------------------------------------------------------------
# Step 6: Interactive dashboard (Dash + Plotly)
# ------------------------------------------------------------------

from dash import Dash, dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px

app = Dash(__name__)

time_fig = px.bar(
    df_time, x='year', y='total',
    labels={'year': 'Year', 'total': 'Questions Asked'},
    title='Neo4j-Tagged Questions by Year'
)

scatter_fig = px.strip(
    df_scores, x='isAccepted', y='score', color='isAccepted',
    labels={'isAccepted': 'Accepted Answer', 'score': 'Answer Score'},
    title='Answer Score Distribution: Accepted vs. Not Accepted'
)

app.layout = html.Div(style={'fontFamily': 'Arial', 'margin': '20px'}, children=[
    html.H2('Stack Overflow Neo4j Q&A Dashboard'),
    html.P(f'{questions_loaded} questions, {answers_loaded} answers, '
           f'{comments_loaded} comments, {users_loaded} users, {tags_loaded} tags loaded.'),

    html.Div(style={'display': 'flex', 'gap': '40px', 'flexWrap': 'wrap'}, children=[
        html.Div(style={'flex': '1', 'minWidth': '320px'}, children=[
            dcc.Graph(id='time-chart', figure=time_fig),
        ]),
        html.Div(style={'flex': '1', 'minWidth': '320px'}, children=[
            dcc.Graph(id='score-chart', figure=scatter_fig),
        ]),
    ]),

    html.Label('Filter top-viewed questions by minimum view count:'),
    dcc.Slider(
        id='view-slider',
        min=int(df_top_questions['views'].min()),
        max=int(df_top_questions['views'].max()),
        value=int(df_top_questions['views'].min()),
        tooltip={'placement': 'bottom'}
    ),
    dash_table.DataTable(
        id='question-table',
        columns=[{'name': c, 'id': c} for c in df_top_questions.columns],
        data=df_top_questions.to_dict('records'),
        page_size=10,
        style_cell={'textAlign': 'left', 'padding': '6px'},
        style_header={'fontWeight': 'bold'},
    ),
])

@app.callback(
    Output('question-table', 'data'),
    Input('view-slider', 'value')
)
def filterByViews(min_views):
    filtered = df_top_questions[df_top_questions['views'] >= min_views]
    return filtered.to_dict('records')

# Run inline inside the Jupyter Notebook output cell. Port 8060 is
# used instead of the more common 8050 to avoid a conflict if an
# earlier Module Eight dashboard kernel is still running in the
# background on this machine.
app.run(jupyter_mode='inline', port=8060)
